<a href="https://colab.research.google.com/github/Adrian-Ang012/Eskwelab-Aviation-Revenue-Ancillary-Attach-Pricing-Simulation/blob/main/data_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Behavioral Synthetic Data Engine (BSDE)

This notebook generates **synthetic airline ancillary transaction datasets** designed for experimentation, modeling, and analytics.

The generator simulates realistic passenger booking behavior and ancillary purchasing decisions using a behavioral framework.

The engine produces three relational tables:

- **dim_bookings** – booking-level passenger and flight context  
- **fact_offers** – ancillary offers shown during the booking flow  
- **fact_conversions** – purchase outcomes of each offer

These datasets can be used for:

- Revenue management modeling
- Price elasticity experiments
- Conversion analysis
- Machine learning training
- Analytics pipeline testing

---

## Dataset Size

The generator can scale from: 1 rows to 1 billion rows (1 billion rows will take longer to generate)

Batching and vectorized generation ensure memory-efficient processing.

---

## Usage
Run the Notebook using the "Run all" button.

In [13]:
from __future__ import annotations

import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Literal, Optional, Tuple

import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    pa = None
    pq = None


OutputFormat = Literal["parquet", "csv"]


@dataclass(frozen=True)
class GeneratorConfig:
    total_rows: int = 1_000_000
    batch_size: Optional[int] = None
    output_dir: str = "output"
    output_format: OutputFormat = "parquet"
    compression: str = "zstd"
    seed: int = 42
    start_timestamp: str = "2025-01-01T00:00:00Z"
    duplicate_rate: float = 0.002
    include_debug_columns: bool = False


class _BatchWriter:
    def __init__(
        self,
        table_name: str,
        output_dir: str,
        output_format: OutputFormat,
        compression: str = "zstd",
    ) -> None:
        self.table_name = table_name
        self.output_dir = Path(output_dir)
        self.output_format = output_format
        self.compression = compression
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self._parquet_writer: Optional["pq.ParquetWriter"] = None
        self._csv_initialized = False
        self._path = self.output_dir / (
            f"{self.table_name}.parquet"
            if self.output_format == "parquet"
            else f"{self.table_name}.csv"
        )

        if self.output_format == "parquet" and pq is None:
            raise ImportError("pyarrow is required for parquet output. Install: pip install pyarrow")

    @property
    def path(self) -> Path:
        return self._path

    def write(self, df: pd.DataFrame) -> None:
        if df.empty:
            return
        if self.output_format == "parquet":
            self._write_parquet(df)
        else:
            self._write_csv(df)

    def _write_parquet(self, df: pd.DataFrame) -> None:
        table = pa.Table.from_pandas(df, preserve_index=False)
        if self._parquet_writer is None:
            self._parquet_writer = pq.ParquetWriter(
                where=str(self._path),
                schema=table.schema,
                compression=self.compression,
            )
        self._parquet_writer.write_table(table)

    def _write_csv(self, df: pd.DataFrame) -> None:
        df.to_csv(
            self._path,
            mode="a",
            index=False,
            header=not self._csv_initialized,
        )
        self._csv_initialized = True

    def close(self) -> None:
        if self._parquet_writer is not None:
            self._parquet_writer.close()
            self._parquet_writer = None


class AncillaryDataGenerator:
    PERSONAS = np.array(["Business", "Leisure", "VFR"], dtype=object)
    PERSONA_PROBS = np.array([0.20, 0.50, 0.30], dtype=np.float64)

    CHANNELS = np.array(["Direct", "OTA", "GDS"], dtype=object)
    CHANNEL_PROBS = np.array([0.55, 0.30, 0.15], dtype=np.float64)

    ANCILLARY_TYPES = np.array(["Baggage", "Seat", "Meal"], dtype=object)
    ANCILLARY_DISPLAY_RANK = np.array([1, 2, 3], dtype=np.int8)

    def __init__(self, config: Optional[GeneratorConfig] = None, seed: int = 42) -> None:
        self.config = config or GeneratorConfig(seed=seed)
        self.rng = np.random.default_rng(self.config.seed)
        self.start_ts = pd.Timestamp(self.config.start_timestamp, tz="UTC")

    def _compute_batch_size(self, n: int) -> int:
        if n <= 100_000:
            return n
        if n <= 10_000_000:
            return 100_000
        if n <= 200_000_000:
            return 1_000_000
        return 5_000_000

    def number_of_rows(self, n: int) -> Dict[str, str]:
        if n <= 0:
            raise ValueError("n must be > 0")

        batch_size = self.config.batch_size or self._compute_batch_size(n)
        total_batches = math.ceil(n / batch_size)

        bookings_writer = _BatchWriter(
            table_name="dim_bookings",
            output_dir=self.config.output_dir,
            output_format=self.config.output_format,
            compression=self.config.compression,
        )
        offers_writer = _BatchWriter(
            table_name="fact_offers",
            output_dir=self.config.output_dir,
            output_format=self.config.output_format,
            compression=self.config.compression,
        )
        conversions_writer = _BatchWriter(
            table_name="fact_conversions",
            output_dir=self.config.output_dir,
            output_format=self.config.output_format,
            compression=self.config.compression,
        )

        next_offer_id = 0
        next_conversion_id = 0

        try:
            print(f"Rows: {n:,}")
            print(f"Batch size: {batch_size:,}")
            print(f"Batches: {total_batches:,}")

            for batch_idx in range(total_batches):
                start_booking_id = batch_idx * batch_size
                current_size = min(batch_size, n - start_booking_id)

                print(f"Generating batch {batch_idx + 1}/{total_batches} ({current_size:,} rows)")

                bookings_df = self._generate_bookings_batch(start_booking_id, current_size)
                offers_df, next_offer_id = self._generate_offers_batch(bookings_df, next_offer_id)
                conversions_df, next_conversion_id = self._generate_conversions_batch(
                    bookings_df,
                    offers_df,
                    next_conversion_id,
                )

                bookings_writer.write(bookings_df)
                offers_writer.write(offers_df)
                conversions_writer.write(conversions_df)

            return {
                "dim_bookings": str(bookings_writer.path),
                "fact_offers": str(offers_writer.path),
                "fact_conversions": str(conversions_writer.path),
            }
        finally:
            bookings_writer.close()
            offers_writer.close()
            conversions_writer.close()

    def _generate_bookings_batch(self, start_booking_id: int, size: int) -> pd.DataFrame:
        booking_id = np.arange(start_booking_id, start_booking_id + size, dtype=np.int64)

        persona_idx = self.rng.choice(len(self.PERSONAS), size=size, p=self.PERSONA_PROBS)
        channel_idx = self.rng.choice(len(self.CHANNELS), size=size, p=self.CHANNEL_PROBS)

        market_segment = self.PERSONAS[persona_idx]
        channel = self.CHANNELS[channel_idx]

        days_to_dep = self._generate_days_to_dep(persona_idx, size)
        base_fare_amt = self._generate_base_fare(persona_idx, size)
        load_factor = np.round(self.rng.uniform(0.40, 0.98, size=size), 4).astype(np.float32)

        df = pd.DataFrame(
            {
                "booking_id": booking_id,
                "market_segment": pd.Categorical(market_segment, categories=list(self.PERSONAS)),
                "channel": pd.Categorical(channel, categories=list(self.CHANNELS)),
                "days_to_dep": days_to_dep.astype(np.int16),
                "base_fare_amt": np.round(base_fare_amt, 2),
                "load_factor": load_factor,
            }
        )

        if self.config.include_debug_columns:
            df["persona_code"] = persona_idx.astype(np.int8)
            df["channel_code"] = channel_idx.astype(np.int8)

        return df

    def _generate_days_to_dep(self, persona_idx: np.ndarray, size: int) -> np.ndarray:
        lam = np.select(
            [persona_idx == 0, persona_idx == 1, persona_idx == 2],
            [4.0, 36.0, 18.0],
            default=14.0,
        )
        return np.clip(self.rng.poisson(lam, size=size), 0, 365)

    def _generate_base_fare(self, persona_idx: np.ndarray, size: int) -> np.ndarray:
        business = self.rng.uniform(550, 1500, size=size)
        leisure = self.rng.uniform(70, 420, size=size)
        vfr = self.rng.uniform(120, 700, size=size)
        return np.select(
            [persona_idx == 0, persona_idx == 1, persona_idx == 2],
            [business, leisure, vfr],
            default=leisure,
        )

    def _generate_offers_batch(
        self,
        bookings_df: pd.DataFrame,
        start_offer_id: int,
    ) -> Tuple[pd.DataFrame, int]:
        n = len(bookings_df)
        repeat_n = len(self.ANCILLARY_TYPES)

        booking_id = np.repeat(bookings_df["booking_id"].to_numpy(dtype=np.int64), repeat_n)
        market_segment = np.repeat(bookings_df["market_segment"].astype(str).to_numpy(), repeat_n)
        channel = np.repeat(bookings_df["channel"].astype(str).to_numpy(), repeat_n)
        days_to_dep = np.repeat(bookings_df["days_to_dep"].to_numpy(dtype=np.int16), repeat_n)
        base_fare_amt = np.repeat(bookings_df["base_fare_amt"].to_numpy(dtype=np.float64), repeat_n)
        load_factor = np.repeat(bookings_df["load_factor"].to_numpy(dtype=np.float64), repeat_n)

        ancillary_type = np.tile(self.ANCILLARY_TYPES, n)
        display_rank = np.tile(self.ANCILLARY_DISPLAY_RANK, n)
        offer_id = np.arange(start_offer_id, start_offer_id + (n * repeat_n), dtype=np.int64)

        inventory_rem = self._generate_inventory(ancillary_type, load_factor)
        offer_price = self._generate_offer_prices(
            ancillary_type=ancillary_type,
            market_segment=market_segment,
            channel=channel,
            base_fare_amt=base_fare_amt,
            load_factor=load_factor,
            days_to_dep=days_to_dep,
        )

        offer_price = np.round(offer_price, 2)
        price_fare_ratio = np.round(offer_price / np.maximum(base_fare_amt, 1.0), 4)

        offers_df = pd.DataFrame(
            {
                "offer_id": offer_id,
                "booking_id": booking_id,
                "ancillary_type": pd.Categorical(
                    ancillary_type,
                    categories=list(self.ANCILLARY_TYPES),
                    ordered=True,
                ),
                "offer_price": offer_price,
                "display_rank": display_rank.astype(np.int8),
                "inventory_rem": inventory_rem.astype(np.int32),
                "price_fare_ratio": price_fare_ratio.astype(np.float32),
            }
        )

        next_offer_id = int(offer_id[-1] + 1) if len(offer_id) else start_offer_id
        return offers_df, next_offer_id

    def _generate_inventory(self, ancillary_type: np.ndarray, load_factor: np.ndarray) -> np.ndarray:
        base_inventory = np.select(
            [ancillary_type == "Baggage", ancillary_type == "Seat", ancillary_type == "Meal"],
            [120, 40, 60],
            default=50,
        ).astype(np.int32)

        scarcity_penalty = np.floor((load_factor - 0.40) / 0.58 * 28).astype(np.int32)
        noise = self.rng.integers(-2, 3, size=len(ancillary_type), dtype=np.int32)
        inventory = np.maximum(base_inventory - scarcity_penalty + noise, 0)
        return inventory

    def _generate_offer_prices(
        self,
        ancillary_type: np.ndarray,
        market_segment: np.ndarray,
        channel: np.ndarray,
        base_fare_amt: np.ndarray,
        load_factor: np.ndarray,
        days_to_dep: np.ndarray,
    ) -> np.ndarray:
        type_multiplier = np.select(
            [ancillary_type == "Baggage", ancillary_type == "Seat", ancillary_type == "Meal"],
            [0.12, 0.09, 0.05],
            default=0.05,
        )

        persona_multiplier = np.select(
            [market_segment == "Business", market_segment == "Leisure", market_segment == "VFR"],
            [1.18, 0.88, 1.00],
            default=1.00,
        )

        channel_multiplier = np.select(
            [channel == "Direct", channel == "OTA", channel == "GDS"],
            [1.00, 1.10, 1.06],
            default=1.00,
        )

        urgency_multiplier = 1.0 + np.clip((30 - days_to_dep) / 100.0, 0.0, 0.22)
        scarcity_multiplier = 1.0 + np.clip((load_factor - 0.72) * 0.45, 0.0, 0.14)
        noise = self.rng.uniform(0.94, 1.06, size=len(ancillary_type))

        price = (
            base_fare_amt
            * type_multiplier
            * persona_multiplier
            * channel_multiplier
            * urgency_multiplier
            * scarcity_multiplier
            * noise
        )

        floors = np.select(
            [ancillary_type == "Baggage", ancillary_type == "Seat", ancillary_type == "Meal"],
            [18.0, 8.0, 5.0],
            default=5.0,
        )

        return np.maximum(price, floors)

    def _generate_conversions_batch(
        self,
        bookings_df: pd.DataFrame,
        offers_df: pd.DataFrame,
        start_conversion_id: int,
    ) -> Tuple[pd.DataFrame, int]:
        n = len(bookings_df)

        offer_id = offers_df["offer_id"].to_numpy(dtype=np.int64)
        ancillary_type = offers_df["ancillary_type"].astype(str).to_numpy()
        offer_price = offers_df["offer_price"].to_numpy(dtype=np.float64)
        display_rank = offers_df["display_rank"].to_numpy(dtype=np.int8)
        inventory_rem = offers_df["inventory_rem"].to_numpy(dtype=np.int32)
        price_fare_ratio = offers_df["price_fare_ratio"].to_numpy(dtype=np.float64)

        bookings_sorted = bookings_df.sort_values("booking_id", kind="stable").reset_index(drop=True)
        market_segment = bookings_sorted["market_segment"].astype(str).to_numpy()
        channel = bookings_sorted["channel"].astype(str).to_numpy()
        days_to_dep = bookings_sorted["days_to_dep"].to_numpy(dtype=np.int16)
        base_fare_amt = bookings_sorted["base_fare_amt"].to_numpy(dtype=np.float64)
        load_factor = bookings_sorted["load_factor"].to_numpy(dtype=np.float64)

        ancillary_2d = ancillary_type.reshape(n, 3)
        offer_price_2d = offer_price.reshape(n, 3)
        ratio_2d = price_fare_ratio.reshape(n, 3)
        inventory_2d = inventory_rem.reshape(n, 3)
        rank_2d = display_rank.reshape(n, 3)

        purchased_2d = np.zeros((n, 3), dtype=np.int8)

        for j in range(3):
            p = self._purchase_probability(
                ancillary_type=ancillary_2d[:, j],
                market_segment=market_segment,
                channel=channel,
                days_to_dep=days_to_dep,
                base_fare_amt=base_fare_amt,
                load_factor=load_factor,
                offer_price=offer_price_2d[:, j],
                display_rank=rank_2d[:, j],
                inventory_rem=inventory_2d[:, j],
                price_fare_ratio=ratio_2d[:, j],
                prior_purchases=purchased_2d[:, :j],
            )
            purchased_2d[:, j] = (self.rng.random(n) < p).astype(np.int8)

        is_purchased = purchased_2d.reshape(-1)
        basket_value_per_booking = (offer_price_2d * purchased_2d).sum(axis=1)
        basket_value = np.repeat(np.round(basket_value_per_booking, 2), 3)

        base_seconds = self.rng.integers(0, 90 * 24 * 60 * 60, size=n, dtype=np.int64)
        booking_ts = self.start_ts + pd.to_timedelta(base_seconds, unit="s")
        event_ts = np.repeat(booking_ts.to_numpy(dtype="datetime64[ns]"), 3)
        within_session_ms = np.tile(np.array([0, 250, 500], dtype=np.int64), n)
        event_ts = event_ts + within_session_ms.astype("timedelta64[ms]")

        conversion_id = np.arange(
            start_conversion_id,
            start_conversion_id + len(offer_id),
            dtype=np.int64,
        )

        conversions_df = pd.DataFrame(
            {
                "conversion_id": conversion_id,
                "offer_id": offer_id,
                "is_purchased": is_purchased.astype(np.int8),
                "timestamp": pd.to_datetime(event_ts, utc=True),
                "basket_value": basket_value.astype(np.float32),
            }
        )

        conversions_df = self._inject_duplicates(conversions_df)
        next_conversion_id = int(conversion_id[-1] + 1) if len(conversion_id) else start_conversion_id
        return conversions_df, next_conversion_id

    def _purchase_probability(
        self,
        ancillary_type: np.ndarray,
        market_segment: np.ndarray,
        channel: np.ndarray,
        days_to_dep: np.ndarray,
        base_fare_amt: np.ndarray,
        load_factor: np.ndarray,
        offer_price: np.ndarray,
        display_rank: np.ndarray,
        inventory_rem: np.ndarray,
        price_fare_ratio: np.ndarray,
        prior_purchases: np.ndarray,
    ) -> np.ndarray:
        type_bias = np.select(
            [ancillary_type == "Baggage", ancillary_type == "Seat", ancillary_type == "Meal"],
            [0.55, 0.20, -0.05],
            default=0.0,
        )

        persona_bias = np.select(
            [market_segment == "Business", market_segment == "Leisure", market_segment == "VFR"],
            [0.40, -0.10, 0.18],
            default=0.0,
        )

        baggage_vfr_bonus = ((market_segment == "VFR") & (ancillary_type == "Baggage")).astype(np.float64) * 0.42
        seat_business_bonus = ((market_segment == "Business") & (ancillary_type == "Seat")).astype(np.float64) * 0.34
        baggage_leisure_bonus = ((market_segment == "Leisure") & (ancillary_type == "Baggage")).astype(np.float64) * 0.18

        channel_term = np.select(
            [channel == "Direct", channel == "OTA", channel == "GDS"],
            [0.04, -0.10, -0.03],
            default=0.0,
        )

        urgency_term = np.clip((25 - days_to_dep) / 25.0, -1.0, 1.0) * 0.22
        scarcity_term = np.where(inventory_rem < 8, 0.24, 0.0) + np.clip((load_factor - 0.78) * 0.55, 0.0, 0.20)
        rank_term = -0.15 * (display_rank - 1)

        threshold = np.select(
            [market_segment == "Business", market_segment == "Leisure", market_segment == "VFR"],
            [0.28, 0.24, 0.25],
            default=0.25,
        )

        below_threshold_penalty = -3.4 * price_fare_ratio
        above_threshold_penalty = -3.4 * threshold - 18.0 * np.maximum(price_fare_ratio - threshold, 0.0)
        price_term = np.where(price_fare_ratio <= threshold, below_threshold_penalty, above_threshold_penalty)

        if prior_purchases.shape[1] == 0:
            reinforcement_term = np.zeros(len(offer_price), dtype=np.float64)
        else:
            reinforcement_term = prior_purchases.sum(axis=1).astype(np.float64) * 0.30

        epsilon = self.rng.normal(0.0, 0.12, size=len(offer_price))

        utility = (
            -0.95
            + type_bias
            + persona_bias
            + baggage_vfr_bonus
            + seat_business_bonus
            + baggage_leisure_bonus
            + channel_term
            + urgency_term
            + scarcity_term
            + rank_term
            + reinforcement_term
            + price_term
            + epsilon
        )

        return 1.0 / (1.0 + np.exp(-np.clip(utility, -20, 20)))

    def _inject_duplicates(self, conversions_df: pd.DataFrame) -> pd.DataFrame:
        if self.config.duplicate_rate <= 0 or conversions_df.empty:
            return conversions_df

        duplicate_count = int(len(conversions_df) * self.config.duplicate_rate)
        if duplicate_count == 0:
            return conversions_df

        chosen_idx = self.rng.choice(len(conversions_df), size=duplicate_count, replace=False)
        dup_df = conversions_df.iloc[chosen_idx].copy()
        jitter_ms = self.rng.integers(-20, 21, size=duplicate_count, dtype=np.int64)
        dup_df["timestamp"] = dup_df["timestamp"] + pd.to_timedelta(jitter_ms, unit="ms")
        return pd.concat([conversions_df, dup_df], ignore_index=True)



## 🗺️ RUNNING THE GENERATOR
To run the dataset generator

1. Instantiate first the main function AncillaryDataSetGenerator(CONFIG)
2. Determine the number of rows you want to generate as the variable n.
(n = 10, 1000, 10000000, 1000000)
3. use the function number_of_rows(n)

## 🗺️ EXAMPLE

1. engine - AncillaryDataGenerator(CONFIG)
2. engine.number_of_rows(n) ; where n is the number of rows

This class can take a paramater n where n is the number of rows. engine.number_of_rows(n)




### 🗺️ Dataset Export and Persistence

Now that the simulation is complete, the generated datasets must be exported. This step persists the processed **Bookings**, **Offers**, and **Conversions** tables into CSV files.

These files will serve as the primary input for the **Showcase & Analysis Notebook**, allowing for Exploratory Data Analysis (EDA) and machine learning model validation without the need to re-run the resource-intensive simulation engine.



In [16]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

local_output = "/content/bsde_output/"
drive_output = "/content/drive/MyDrive/Ian - Aviation revenue/"

os.makedirs(local_output, exist_ok=True)
os.makedirs(drive_output, exist_ok=True)

config = GeneratorConfig(
    total_rows=10_000_000,
    output_dir=local_output,
    output_format="parquet",
    compression="zstd",
    seed=42,
)

engine = AncillaryDataGenerator(config)
paths = engine.number_of_rows(10_000_000)

for filename in [
    "dim_bookings.parquet",
    "fact_offers.parquet",
    "fact_conversions.parquet",
]:
    src = os.path.join(local_output, filename)
    dst = os.path.join(drive_output, filename)

    if os.path.exists(dst):
        os.remove(dst)

    shutil.copy2(src, dst)

print("Files copied to Google Drive successfully.")
print("Local files:")
print(paths)

print("\nDrive files:")
print({
    "dim_bookings": os.path.join(drive_output, "dim_bookings.parquet"),
    "fact_offers": os.path.join(drive_output, "fact_offers.parquet"),
    "fact_conversions": os.path.join(drive_output, "fact_conversions.parquet"),
})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Rows: 10,000,000
Batch size: 100,000
Batches: 100
Generating batch 1/100 (100,000 rows)
Generating batch 2/100 (100,000 rows)
Generating batch 3/100 (100,000 rows)
Generating batch 4/100 (100,000 rows)
Generating batch 5/100 (100,000 rows)
Generating batch 6/100 (100,000 rows)
Generating batch 7/100 (100,000 rows)
Generating batch 8/100 (100,000 rows)
Generating batch 9/100 (100,000 rows)
Generating batch 10/100 (100,000 rows)
Generating batch 11/100 (100,000 rows)
Generating batch 12/100 (100,000 rows)
Generating batch 13/100 (100,000 rows)
Generating batch 14/100 (100,000 rows)
Generating batch 15/100 (100,000 rows)
Generating batch 16/100 (100,000 rows)
Generating batch 17/100 (100,000 rows)
Generating batch 18/100 (100,000 rows)
Generating batch 19/100 (100,000 rows)
Generating batch 20/100 (100,000 rows)
Generating batch 21/100 (100,000 rows)
Generating 